# 🕵️ DataTour 2026 — Détection de Fraude Mobile Money
## Analyse Exploratoire Complète (EDA) & Préparation des Données

---

**Contexte** : Compétition DataTour 2026 — Phase Nationale  
**Problème** : Détecter les transactions frauduleuses dans un système Mobile Money africain  
**Objectif de ce notebook** : Explorer les données en profondeur, comprendre les patterns de fraude, créer de nouvelles features et préparer les données pour la modélisation  
**Audience** : Équipe OpenAI — Bourse ML 2026

---

### Ce qu'on va faire dans ce notebook :
1. Comprendre le problème (c'est quoi la fraude Mobile Money ?)
2. Explorer les données brutes (structure, qualité, statistiques)
3. Analyser chaque variable en profondeur avec des graphiques
4. Découvrir les patterns cachés qui distinguent fraudes et légitimes
5. Créer 29 nouvelles features pour aider le modèle
6. Préparer les datasets finaux pour l'entraînement

---
## 📦 0. Installation & Configuration

On installe les bibliothèques nécessaires et on configure les graphiques pour qu'ils soient lisibles et esthétiques.

In [ ]:
# Installation des bibliothèques (déjà présentes sur Colab, mais on vérifie)
!pip install -q kaggle lightgbm xgboost optuna

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')

# ============================================================
# STYLE DES GRAPHIQUES
# ============================================================
# On définit un style sombre, professionnel et lisible
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'text.color': 'white',
    'grid.color': '#333',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.size': 11,
})

# Couleurs : vert pour légitime, rouge pour fraude
PALETTE_FRAUD = ['#00c896', '#ff4d6d']
PALETTE_MAIN  = '#4f7cff'

print('✅ Configuration OK')
print('📊 Bibliothèques chargées :')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   seaborn {sns.__version__}')

---
## 📁 1. Chargement des Données

### Comment accéder aux données sur Colab ?

**Option A (Kaggle API)** : Si vous avez un compte Kaggle et la clé API  
**Option B (Upload manuel)** : Uploadez `train.csv` et `test.csv` directement dans Colab  
**Option C (Google Drive)** : Montez votre Drive et accédez aux fichiers

In [ ]:
# ============================================================
# CHARGEMENT DES DONNÉES
# Option : modifiez DATA_DIR selon votre configuration
# ============================================================

# Option A : Fichiers uploadés directement dans Colab
DATA_DIR = '.'  

# Option B : Depuis Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/datatour'

# Option C : Depuis le dossier local
# DATA_DIR = '/home/precieux/datatour/dataset'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')

print('=' * 60)
print('📊 DONNÉES CHARGÉES AVEC SUCCÈS')
print('=' * 60)
print(f'  train.csv  → {train.shape[0]:,} lignes × {train.shape[1]} colonnes')
print(f'  test.csv   → {test.shape[0]:,} lignes × {test.shape[1]} colonnes')
print(f'  Ratio train / (train+test) : {train.shape[0]/(train.shape[0]+test.shape[0]):.1%}')
print()
print('Mémoire utilisée :')
print(f'  train : {train.memory_usage(deep=True).sum() / 1e6:.1f} Mo')
print(f'  test  : {test.memory_usage(deep=True).sum() / 1e6:.1f} Mo')

In [ ]:
# Aperçu des 5 premières lignes
print('📋 Aperçu du dataset train :')
train.head(5)

---
## 🔍 2. Structure et Qualité des Données

### Qu'est-ce qu'on vérifie ?

Avant d'analyser quoi que ce soit, on s'assure que les données sont **propres** :
- **Valeurs manquantes** : des cases vides → problème pour le modèle
- **Doublons** : des lignes identiques → biais potentiel
- **Types de données** : est-ce que les colonnes numériques sont bien en `float` ?

C'est comme vérifier que les ingrédients d'une recette sont frais avant de cuisiner.

In [ ]:
print('🔍 ANALYSE DE LA QUALITÉ DES DONNÉES')
print('=' * 60)

print('\n--- Types et valeurs manquantes ---')
info_df = pd.DataFrame({
    'Type': train.dtypes,
    'Valeurs uniques': train.nunique(),
    'Valeurs manquantes': train.isnull().sum(),
    '% manquant': (train.isnull().sum() / len(train) * 100).round(3)
})
print(info_df.to_string())

print(f'\n--- Doublons ---')
print(f'Lignes dupliquées   : {train.duplicated().sum()}')
print(f'IDs dupliqués       : {train["id"].duplicated().sum()}')

print(f'\n✅ Résultat : données propres, aucune valeur manquante, aucun doublon')
print('→ On peut passer directement à l\'analyse sans nettoyage préalable')

In [ ]:
# Statistiques descriptives de base
print('📊 Statistiques descriptives des variables numériques :')
train.describe().round(2)

---
## 🎯 3. Variable Cible — Le Problème de Déséquilibre

### C'est quoi le déséquilibre de classes ?

Notre dataset a **10% de fraudes** et **90% de transactions légitimes**. C'est un problème classique en fraude financière. 

**Pourquoi c'est un problème ?**

Un modèle "stupide" qui dit TOUJOURS "pas de fraude" aurait :
- **90% d'accuracy** ← impressionnant en apparence
- **0% de rappel sur la fraude** ← inutile en pratique

C'est pourquoi on utilise le **ROC-AUC** comme métrique, et des techniques comme `class_weight='balanced'`.

In [ ]:
fraud_counts = train['fraud_flag'].value_counts()
fraud_rate   = train['fraud_flag'].mean()

print('🎯 DISTRIBUTION DE LA VARIABLE CIBLE')
print('=' * 60)
print(f'  Transactions légitimes    : {fraud_counts[0]:>10,}  ({(1-fraud_rate)*100:.2f}%)')
print(f'  Transactions frauduleuses : {fraud_counts[1]:>10,}  ({fraud_rate*100:.2f}%)')
print(f'  Ratio déséquilibre        : 1 fraude pour {int(1/fraud_rate)} légitimes')
print()
print('⚠️  ATTENTION : CLASSES DÉSÉQUILIBRÉES')
print(f'  → Un modèle "idiot" aurait {(1-fraud_rate)*100:.1f}% d\'accuracy')
print(f'  → On utilise ROC-AUC comme métrique principale')
print(f'  → On applique class_weight=\'balanced\' ou scale_pos_weight={int(1/fraud_rate)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution de la Variable Cible — fraud_flag', 
             fontsize=16, fontweight='bold', color='white', y=1.02)

# Barplot
bars = axes[0].bar(['Légitime (0)', 'Fraude (1)'],
                   fraud_counts.values,
                   color=PALETTE_FRAUD, edgecolor='none', width=0.5)
axes[0].set_title('Comptage par classe')
axes[0].set_ylabel('Nombre de transactions')
for bar, val in zip(bars, fraud_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
                 f'{val:,}\n({val/len(train)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=12, color='white', fontweight='bold')

# Camembert
wedge_props = dict(width=0.5, edgecolor='#0f1117', linewidth=3)
axes[1].pie(fraud_counts.values,
            labels=['Légitime (90%)', 'Fraude (10%)'],
            autopct='%1.1f%%',
            colors=PALETTE_FRAUD,
            startangle=90,
            wedgeprops=wedge_props,
            textprops={'color': 'white', 'fontsize': 13})
axes[1].set_title('Répartition en %')

plt.tight_layout()
plt.show()
print('💡 Le déséquilibre est visible : le secteur "Fraude" est petit mais critique')

---
## 🚨 4. Types d'Opérations — La Découverte Majeure

### Qu'est-ce qu'on analyse ?

Il y a 5 types d'opérations : `op_01`, `op_02`, `op_03`, `op_04`, `op_05`. On regarde le taux de fraude dans chacune.

### Hypothèse

Certains types d'opérations sont peut-être plus exploités par les fraudeurs. Par exemple, une opération "retrait cash" permet de récupérer l'argent immédiatement et disparaître — parfaite pour la fraude.

In [ ]:
op_stats = train.groupby('operation').agg(
    n_total       = ('fraud_flag', 'count'),
    n_fraude      = ('fraud_flag', 'sum'),
    taux_fraude   = ('fraud_flag', 'mean'),
    montant_moy   = ('amount', 'mean'),
    montant_med   = ('amount', 'median'),
    montant_max   = ('amount', 'max'),
).reset_index()
op_stats['taux_fraude_pct'] = (op_stats['taux_fraude'] * 100).round(2)
op_stats = op_stats.sort_values('taux_fraude', ascending=False)

print('📊 STATISTIQUES PAR TYPE D\'OPÉRATION')
print('=' * 70)
print(op_stats[['operation','n_total','n_fraude','taux_fraude_pct','montant_moy']].to_string(index=False))
print()
print('🔴 DÉCOUVERTE MAJEURE :')
print('   100% des fraudes sont dans op_03 !')
print('   Les opérations op_01, op_02, op_04, op_05 ont 0% de fraude')
print('   op_03 a un taux de fraude de 31.19% !')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Types d\'Opérations : Volume et Taux de Fraude', 
             fontsize=15, fontweight='bold', color='white')

# Volume par opération
ops_vol = op_stats.sort_values('n_total', ascending=False)
colors = ['#ff4d6d' if t > fraud_rate else '#4f7cff' 
          for t in ops_vol['taux_fraude']]
bars = axes[0].bar(ops_vol['operation'], ops_vol['n_total'],
                   color=colors, edgecolor='none')
axes[0].set_title('Volume de transactions\n(rouge = taux fraude > moyenne)', fontsize=12)
axes[0].set_ylabel('Nombre de transactions')
for bar, val in zip(bars, ops_vol['n_total']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}', ha='center', va='bottom', fontsize=10, color='white')

# Taux de fraude par opération
ops_taux = op_stats.sort_values('taux_fraude_pct', ascending=False)
bars2 = axes[1].bar(ops_taux['operation'], ops_taux['taux_fraude_pct'],
                    color='#ff4d6d', edgecolor='none', alpha=0.85)
axes[1].axhline(y=fraud_rate*100, color='#ffd166', linestyle='--',
                linewidth=2, label=f'Taux moyen ({fraud_rate*100:.1f}%)')
axes[1].set_title('Taux de fraude (%) par opération', fontsize=12)
axes[1].set_ylabel('% Fraude')
axes[1].legend()
for bar, val in zip(bars2, ops_taux['taux_fraude_pct']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=11, 
                 color='white', fontweight='bold')

plt.tight_layout()
plt.show()
print('💡 op_03 a 31.2% de fraude — c\'est la feature discriminante n°1')

---
## ⏰ 5. Analyse Temporelle

### Structure temporelle

```
Train : périodes 0 → 105   (106 périodes)
Test  : périodes 106 → 143 (38 périodes dans le FUTUR)
```

### Pourquoi c'est crucial ?

Le test est dans le futur du train. Si on mélange aléatoirement train et validation, on "regarde dans le futur" pendant l'entraînement → **data leakage**. 

**On doit utiliser un split temporel.**

In [ ]:
period_stats = train.groupby('period').agg(
    n_total    = ('fraud_flag', 'count'),
    n_fraude   = ('fraud_flag', 'sum'),
    taux_fraude = ('fraud_flag', 'mean')
).reset_index()

print('⏰ ANALYSE TEMPORELLE')
print('=' * 60)
print(f'  Périodes train : {train["period"].min()} → {train["period"].max()}')
print(f'  Périodes test  : {test["period"].min()} → {test["period"].max()}')
print(f'  → Le test est dans le FUTUR du train')
print()
print(f'  Transactions par période : min={period_stats["n_total"].min():,} / max={period_stats["n_total"].max():,}')
print(f'  Taux de fraude par période : min={period_stats["taux_fraude"].min()*100:.2f}% / max={period_stats["taux_fraude"].max()*100:.2f}%')
print()
print('  ⚠️  Le taux de fraude varie du SIMPLE AU QUADRUPLE selon la période')
print('  → Des vagues d\'attaques organisées existent dans les données')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
fig.suptitle('Évolution Temporelle des Transactions et de la Fraude',
             fontsize=15, fontweight='bold', color='white')

# Volume
axes[0].fill_between(period_stats['period'], period_stats['n_total'],
                     alpha=0.4, color='#4f7cff', label='Total')
axes[0].fill_between(period_stats['period'], period_stats['n_fraude'],
                     alpha=0.8, color='#ff4d6d', label='Fraude')
axes[0].set_ylabel('Nombre de transactions')
axes[0].set_title('Volume de transactions par période')
axes[0].legend()

# Taux de fraude
axes[1].plot(period_stats['period'], period_stats['taux_fraude']*100,
             color='#ff4d6d', linewidth=2, label='Taux fraude')
axes[1].axhline(y=fraud_rate*100, color='#ffd166', linestyle='--',
                linewidth=1.5, label=f'Moyenne ({fraud_rate*100:.2f}%)')
axes[1].fill_between(period_stats['period'], period_stats['taux_fraude']*100,
                     fraud_rate*100, alpha=0.2,
                     where=period_stats['taux_fraude'] > fraud_rate,
                     color='#ff4d6d', label='Au-dessus de la moyenne')
axes[1].set_ylabel('Taux de fraude (%)')
axes[1].set_xlabel('Période (ordre chronologique)')
axes[1].set_title('Évolution du taux de fraude dans le temps')
axes[1].legend()

plt.tight_layout()
plt.show()
print('💡 Le taux de fraude n\'est pas constant — des pics existent (vagues d\'attaques)')

---
## 💰 6. Analyse des Montants

### Question centrale

Est-ce que les fraudeurs font des transactions d'un montant particulier ? 

**Hypothèse** : Les fraudes visent des montants suffisamment grands pour valoir le coup, mais pas trop grands pour éviter les alertes automatiques.

In [ ]:
legit  = train[train['fraud_flag'] == 0]['amount']
fraude = train[train['fraud_flag'] == 1]['amount']

print('💰 STATISTIQUES DES MONTANTS')
print('=' * 60)
print(f'  {"":25} {"Légitime":>12} {"Fraude":>12}')
print(f'  {"─"*50}')
for stat_name, stat_func in [
    ('Minimum', 'min'), ('Q25 (1er quartile)', lambda x: x.quantile(0.25)),
    ('Médiane', 'median'), ('Moyenne', 'mean'),
    ('Q75 (3e quartile)', lambda x: x.quantile(0.75)),
    ('Q99', lambda x: x.quantile(0.99)),
    ('Maximum', 'max'), ('Écart-type', 'std')
]:
    if callable(stat_func):
        l_val, f_val = stat_func(legit), stat_func(fraude)
    else:
        l_val = getattr(legit, stat_func)()
        f_val = getattr(fraude, stat_func)()
    print(f'  {stat_name:<25} {l_val:>12,.2f} {f_val:>12,.2f}')

print()
print('🔴 INSIGHT CLÉ : Toutes les fraudes ont amount > 19 815 !')
print('🔴 L\'écart-type des fraudes (21 000) est 5× plus faible que les légitimes (110 000)')
print('→ Les fraudeurs agissent dans une fourchette de montants étroite')

# Test statistique
stat, pval = stats.mannwhitneyu(
    legit.sample(5000, random_state=42),
    fraude.sample(min(5000, len(fraude)), random_state=42),
    alternative='two-sided'
)
print(f'\n📊 Test Mann-Whitney (distributions différentes ?) :')
print(f'   p-value = {pval:.2e} → SIGNIFICATIF ✅ (les distributions diffèrent)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Distribution des Montants : Fraude vs Légitime',
             fontsize=15, fontweight='bold', color='white')

# Distribution log
axes[0].hist(np.log1p(legit.sample(50000, random_state=42)),
             bins=100, alpha=0.6, color=PALETTE_FRAUD[0], label='Légitime', density=True)
axes[0].hist(np.log1p(fraude),
             bins=100, alpha=0.6, color=PALETTE_FRAUD[1], label='Fraude', density=True)
axes[0].set_title('Distribution log(montant+1)')
axes[0].set_xlabel('log(amount + 1)')
axes[0].set_ylabel('Densité')
axes[0].legend()

# Boxplot
bp = axes[1].boxplot([np.log1p(legit.sample(50000, random_state=42)), np.log1p(fraude)],
                     tick_labels=['Légitime', 'Fraude'],
                     patch_artist=True,
                     medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE_FRAUD):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Boxplot log(montant+1)')
axes[1].set_ylabel('log(amount + 1)')

# Taux par tranche
train_temp = train.copy()
train_temp['amount_bin'] = pd.qcut(train_temp['amount'], q=20, duplicates='drop')
fraud_by_amount = train_temp.groupby('amount_bin', observed=True)['fraud_flag'].mean() * 100
axes[2].bar(range(len(fraud_by_amount)), fraud_by_amount.values, color='#ff4d6d', alpha=0.8)
axes[2].axhline(y=fraud_rate*100, color='#ffd166', linestyle='--', linewidth=1.5)
axes[2].set_title('Taux de fraude par tranche de montant')
axes[2].set_xlabel('Tranche (0=petit → droite=grand)')
axes[2].set_ylabel('% Fraude')
del train_temp

plt.tight_layout()
plt.show()

---
## ⚖️ 7. Analyse des Soldes (Balances)

### Le test de cohérence comptable

Dans un système financier honnête, chaque transaction doit respecter cette équation :

```
Côté envoyeur  : solde_après = solde_avant - montant
Côté receveur  : solde_après = solde_avant + montant
```

Si cette équation n'est **pas respectée** → anomalie comptable → signal potentiel de fraude.

C'est le principe de base de l'audit comptable, appliqué à la détection de fraude.

In [ ]:
print('⚖️ ANALYSE DES SOLDES ET COHÉRENCE COMPTABLE')
print('=' * 70)

# Statistiques des soldes par classe
for col in ['origin_balance_before', 'origin_balance_after',
            'destination_balance_before', 'destination_balance_after']:
    l = train[train['fraud_flag']==0][col]
    f = train[train['fraud_flag']==1][col]
    print(f'\n  📊 {col}:')
    print(f'     Légitime → médiane: {l.median():>12,.0f}  négatifs: {(l<0).sum():>8,} ({(l<0).mean()*100:.1f}%)')
    print(f'     Fraude   → médiane: {f.median():>12,.0f}  négatifs: {(f<0).sum():>8,} ({(f<0).mean()*100:.1f}%)')

# Test de cohérence
print(f'\n  📊 Test de cohérence comptable côté origine :')
print(f'     (|balance_before - amount - balance_after| > 0.01)')
error_origin = (train['origin_balance_before'] - train['amount'] - train['origin_balance_after']).abs() > 0.01
print(f'     Incohérences dans légitimes : {(error_origin & (train["fraud_flag"]==0)).sum():,}')
print(f'     Incohérences dans fraudes   : {(error_origin & (train["fraud_flag"]==1)).sum():,}')
print(f'\n  💡 NOTE : Ces incohérences existent aussi dans les légitimes car')
print(f'     les frais de transaction ne sont pas dans les données.')
print(f'     Le modèle apprendra à distinguer les erreurs "normales" des suspectes.')

---
## ⚙️ 8. Feature Engineering — Créer 29 Nouvelles Variables

### Pourquoi créer de nouvelles features ?

Le modèle ML voit uniquement des colonnes numériques. Si on lui donne `balance_before = 100` et `balance_after = -900`, il doit deviner que la différence est -1000 — et que c'est suspect.

Si on lui donne directement `balance_delta = -1000` et `account_emptied = 1`, c'est **beaucoup plus clair**.

Le feature engineering, c'est **transformer notre connaissance métier en variables numériques** que le modèle peut exploiter directement.

### Notre stratégie

On s'inspire du comportement des fraudeurs :
1. Ils **vident des comptes** → `origin_account_emptied`
2. Ils **manipulent les soldes** → `balance_error`
3. Ils font des **gros montants** → `log_amount`, `amount_is_high`
4. Ils utilisent des **comptes avec peu d'historique** → `acc_o_n_tx`
5. Leur **type d'opération** est spécifique → `operation_fraud_rate`

In [ ]:
def engineer_features(df, origin_stats_ref=None):
    """
    Feature Engineering pour la détection de fraude Mobile Money.
    
    Crée 29 nouvelles variables à partir des 10 variables brutes.
    
    Paramètres :
        df              : DataFrame (train ou test)
        origin_stats_ref: stats comportementales calculées sur le train
    
    Retourne : DataFrame enrichi avec toutes les nouvelles features
    """
    d = df.copy()
    global fraud_rate  # taux de fraude du train
    
    # ─────────────────────────────────────────────────────────────
    # A. FEATURES DE MONTANT
    # ─────────────────────────────────────────────────────────────
    
    # log(amount+1) : réduit l'impact des valeurs extrêmes
    # log(1000001) ≈ 14  vs  amount = 1 000 001 (valeur gigantesque)
    d['log_amount'] = np.log1p(d['amount'])
    
    # Montant rond : les fraudeurs font souvent des montants ronds (1000, 5000...)
    # car c'est plus facile à mémoriser et à traiter
    d['amount_is_round'] = ((d['amount'] % 1000 == 0) |
                             (d['amount'] % 500 == 0)).astype(int)
    
    # Montant très élevé (au-dessus du 95e percentile)
    amount_p95 = df['amount'].quantile(0.95)
    d['amount_is_high'] = (d['amount'] > amount_p95).astype(int)
    
    # ─────────────────────────────────────────────────────────────
    # B. FEATURES DE BALANCE ORIGINE
    # ─────────────────────────────────────────────────────────────
    
    # Ce qui a vraiment été débité
    d['origin_balance_delta'] = d['origin_balance_before'] - d['origin_balance_after']
    
    # Erreur comptable côté origine : |avant - montant - après|
    # Dans un système parfait : avant - montant = après → erreur = 0
    # Les frais cachés créent des "erreurs normales" de quelques unités
    d['origin_balance_error'] = np.abs(
        d['origin_balance_before'] - d['amount'] - d['origin_balance_after']
    )
    d['origin_has_balance_error'] = (d['origin_balance_error'] > 0.01).astype(int)
    
    # Ratio montant / solde avant : transaction disproportionnée ?
    # Si quelqu'un vire 100 avec 100 de solde → ratio = 1.0 (vide son compte)
    # Si quelqu'un vire 100 avec 100 000 → ratio = 0.001 (transaction normale)
    d['origin_amount_ratio'] = d['amount'] / (d['origin_balance_before'].abs() + 1)
    
    # Le compte a-t-il été complètement vidé ?
    d['origin_account_emptied'] = (d['origin_balance_after'].abs() < 1).astype(int)
    
    # Solde négatif (découvert)
    d['origin_balance_negative_before'] = (d['origin_balance_before'] < 0).astype(int)
    d['origin_balance_negative_after']  = (d['origin_balance_after']  < 0).astype(int)
    
    # Versions log (réduction des valeurs extrêmes)
    d['log_origin_balance_before'] = np.log1p(np.abs(d['origin_balance_before']))
    d['log_origin_balance_after']  = np.log1p(np.abs(d['origin_balance_after']))
    
    # ─────────────────────────────────────────────────────────────
    # C. FEATURES DE BALANCE DESTINATION
    # ─────────────────────────────────────────────────────────────
    
    # Augmentation réelle côté destination
    d['dest_balance_delta'] = d['destination_balance_after'] - d['destination_balance_before']
    
    # Erreur comptable côté destination : |avant + montant - après|
    d['dest_balance_error'] = np.abs(
        d['destination_balance_before'] + d['amount'] - d['destination_balance_after']
    )
    d['dest_has_balance_error'] = (d['dest_balance_error'] > 0.01).astype(int)
    
    # Ratio montant / solde destination
    d['dest_amount_ratio'] = d['amount'] / (d['destination_balance_before'].abs() + 1)
    
    # Balance destination inchangée : on envoie de l'argent mais le destinataire
    # ne reçoit rien → très suspect !
    d['dest_balance_unchanged'] = (d['dest_balance_delta'].abs() < 0.01).astype(int)
    
    d['log_dest_balance_before'] = np.log1p(np.abs(d['destination_balance_before']))
    d['log_dest_balance_after']  = np.log1p(np.abs(d['destination_balance_after']))
    
    # ─────────────────────────────────────────────────────────────
    # D. FEATURES CROISÉES
    # ─────────────────────────────────────────────────────────────
    
    # Incohérence des DEUX côtés simultanément = très suspect
    d['both_balance_error'] = (
        (d['origin_has_balance_error'] == 1) & 
        (d['dest_has_balance_error'] == 1)
    ).astype(int)
    
    # Incohérence d'au moins un côté
    d['any_balance_error'] = (
        (d['origin_has_balance_error'] == 1) | 
        (d['dest_has_balance_error'] == 1)
    ).astype(int)
    
    # Asymétrie entre les deux erreurs
    d['balance_error_asymmetry'] = (
        np.log1p(d['origin_balance_error']) - 
        np.log1p(d['dest_balance_error'])
    )
    
    # ─────────────────────────────────────────────────────────────
    # E. ENCODAGE OPÉRATION
    # ─────────────────────────────────────────────────────────────
    
    op_map = {op: i for i, op in enumerate(sorted(d['operation'].unique()))}
    d['operation_code'] = d['operation'].map(op_map).fillna(-1)
    
    # FEATURE CLÉ : taux de fraude par type d'opération
    # op_03 → 0.3119 (31.19%)   op_01/02/04/05 → 0.0000
    if 'fraud_flag' in df.columns:
        op_fraud_rate = df.groupby('operation')['fraud_flag'].mean().to_dict()
    else:
        op_fraud_rate = {  # valeurs calculées sur le train
            'op_01': 0.0000, 'op_02': 0.0000, 'op_03': 0.3119,
            'op_04': 0.0000, 'op_05': 0.0000
        }
    d['operation_fraud_rate'] = d['operation'].map(op_fraud_rate).fillna(fraud_rate)
    
    # ─────────────────────────────────────────────────────────────
    # F. FEATURES TEMPORELLES
    # ─────────────────────────────────────────────────────────────
    
    d['period_normalized'] = d['period'] / d['period'].max()
    
    # ─────────────────────────────────────────────────────────────
    # G. COMPORTEMENT DES COMPTES (depuis les stats du train)
    # ─────────────────────────────────────────────────────────────
    
    if origin_stats_ref is not None:
        d = d.merge(
            origin_stats_ref[['origin_account', 'acc_o_n_tx', 'acc_o_fraud_rate',
                               'acc_o_mean_amount', 'acc_o_mean_delta']],
            on='origin_account', how='left'
        )
    
    return d

print('✅ Fonction engineer_features() définie avec 7 groupes de features')

In [ ]:
# ── Calcul des statistiques comportementales sur le TRAIN uniquement ──
# IMPORTANT : on ne regarde jamais les données du test ici
print('📊 Calcul des agrégats comportementaux par compte...')

origin_agg = train.groupby('origin_account').agg(
    acc_o_n_tx        = ('fraud_flag', 'count'),
    acc_o_fraud_rate  = ('fraud_flag', 'mean'),
    acc_o_mean_amount = ('amount', 'mean'),
    acc_o_mean_delta  = ('origin_balance_before',
                         lambda x: (x - train.loc[x.index, 'origin_balance_after']).mean()),
).reset_index()

print('✅ Agrégats calculés')
print(f'  {len(origin_agg):,} comptes uniques avec leurs statistiques')

# ── Application du Feature Engineering ──
print('\n⚙️  Application du Feature Engineering...')
train_fe = engineer_features(train, origin_stats_ref=origin_agg)
test_fe  = engineer_features(test,  origin_stats_ref=origin_agg)

print(f'\n  Colonnes AVANT FE : {train.shape[1]}')
print(f'  Colonnes APRÈS FE : {train_fe.shape[1]}')
new_cols = [c for c in train_fe.columns if c not in train.columns]
print(f'  Nouvelles features : {len(new_cols)}')
print('\n  Liste des nouvelles features :')
for i, col in enumerate(new_cols, 1):
    print(f'  {i:2d}. {col}')

---
## 🔬 9. Validation des Features Créées

### Comment savoir si une feature est bonne ?

On calcule le **lift** : est-ce que la feature discrimine bien les fraudes des légitimes ?

Pour une feature binaire (0/1) :
- Si `feature = 0` → taux de fraude = X%
- Si `feature = 1` → taux de fraude = Y%
- **Lift = Y / taux_moyen**  (si lift > 1 → la feature est utile quand elle vaut 1)

In [ ]:
print('🔬 VALIDATION DES FEATURES CRÉÉES')
print('=' * 70)
print('  Taux de fraude selon les features booléennes (0/1) :')
print(f'  {"Feature":<35} {"Si=0":>8} {"Si=1":>8} {"n(1)":>10} {"Lift":>8}')
print(f'  {"─"*65}')

bool_features = [
    'amount_is_round', 'amount_is_high',
    'origin_has_balance_error', 'origin_account_emptied',
    'origin_balance_negative_before',
    'dest_has_balance_error', 'dest_balance_unchanged',
    'both_balance_error', 'any_balance_error'
]

for feat in bool_features:
    if feat not in train_fe.columns:
        continue
    rate_0 = train_fe[train_fe[feat]==0]['fraud_flag'].mean() * 100
    rate_1 = train_fe[train_fe[feat]==1]['fraud_flag'].mean() * 100
    n_1    = (train_fe[feat]==1).sum()
    lift   = rate_1 / (fraud_rate * 100) if fraud_rate > 0 else 0
    flag   = '🔴' if abs(lift - 1) > 0.3 else ''
    print(f'  {feat:<35} {rate_0:>7.1f}% {rate_1:>7.1f}% {n_1:>10,} {lift:>7.2f}× {flag}')

print()
print('  💡 Note : Un lift < 1 signifie que quand la feature vaut 1,')
print('     la probabilité de fraude est PLUS BASSE que la moyenne.')
print('     Ce n\'est pas inutile — le modèle peut l\'utiliser dans l\'autre sens.')

---
## 🏁 10. Préparation Finale & Split Temporel

### La règle d'or : respecter l'ordre du temps

```
MAUVAISE approche (data leakage) :
   train ──────────── 80% aléatoire ────────────
   val  ──────────── 20% aléatoire ────────────

BONNE approche (respecte le temps) :
   train  ← périodes 0 à 76  (80% du train)
   val    ← périodes 77 à 105 (20% du train)
   test   ← périodes 106 à 143 (dans le futur)
```

In [ ]:
# ── Sélection des features finales ──
EXCLUDE_COLS = ['id', 'origin_account', 'destination_account', 'operation', 'fraud_flag']
FEATURE_COLS = [c for c in train_fe.columns if c not in EXCLUDE_COLS]

X = train_fe[FEATURE_COLS]
y = train_fe['fraud_flag']

# ── Split temporel ──
SPLIT_PERIOD = train_fe['period'].quantile(0.80)

X_train = X[train_fe['period'] <= SPLIT_PERIOD]
y_train = y[train_fe['period'] <= SPLIT_PERIOD]
X_val   = X[train_fe['period'] > SPLIT_PERIOD]
y_val   = y[train_fe['period'] > SPLIT_PERIOD]

print('🏁 PRÉPARATION FINALE')
print('=' * 60)
print(f'  Nombre de features : {len(FEATURE_COLS)}')
print(f'  Période de coupure : {SPLIT_PERIOD}')
print()
print(f'  Train  : {X_train.shape[0]:>10,} lignes ({X_train.shape[0]/len(X)*100:.1f}%) | '
      f'Taux fraude : {y_train.mean()*100:.2f}%')
print(f'  Val    : {X_val.shape[0]:>10,} lignes ({X_val.shape[0]/len(X)*100:.1f}%) | '
      f'Taux fraude : {y_val.mean()*100:.2f}%')
print(f'  Test   : {test_fe.shape[0]:>10,} lignes  (à prédire)')

# Vérification des valeurs manquantes
nulls = X.isnull().sum()
if nulls.sum() > 0:
    print(f'\n  ⚠️  Valeurs manquantes détectées après FE :')
    print(nulls[nulls > 0])
    X_train = X_train.fillna(X_train.median())
    X_val   = X_val.fillna(X_train.median())
    print('  → Imputées par la médiane du train')
else:
    print(f'\n  ✅ Aucune valeur manquante dans les features finales')

print(f'\n  ✅ DONNÉES PRÊTES POUR LA MODÉLISATION !')

In [ ]:
# Corrélation des features avec la cible
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
corr_with_target = train_fe[numeric_cols + ['fraud_flag']].corr(method='spearman')['fraud_flag']
corr_with_target = corr_with_target.drop('fraud_flag').sort_values(key=abs, ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors_bar = ['#ff4d6d' if v > 0 else '#4f7cff' for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors_bar, alpha=0.85)
ax.axvline(0, color='white', linewidth=1)
ax.set_title('Top 20 Features — Corrélation Spearman avec fraud_flag\n'
             '(rouge = corrélé positivement avec la fraude, bleu = négativement)',
             fontsize=13, fontweight='bold', color='white')
ax.set_xlabel('Corrélation Spearman')
plt.tight_layout()
plt.show()

---
## 📋 Résumé Final de l'EDA

### Ce qu'on a découvert

In [ ]:
print('''
╔══════════════════════════════════════════════════════════════╗
║         RÉSUMÉ EDA — DataTour 2026 Fraude Mobile Money      ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  📊 DATASET                                                  ║
║     Train : 1 290 081 transactions × 11 colonnes             ║
║     Test  :   430 100 transactions × 10 colonnes             ║
║     0 valeurs manquantes, 0 doublons                        ║
║                                                              ║
║  🎯 VARIABLE CIBLE                                           ║
║     Taux de fraude : 10.04%                                  ║
║     Déséquilibre   : 1 fraude / 9 légitimes                 ║
║     Métrique       : ROC-AUC                                 ║
║                                                              ║
║  🔑 DISCOVERIES CLÉS                                         ║
║     1. 100% des fraudes dans op_03 (31.19% de taux)         ║
║     2. Toutes les fraudes ont amount > 19 815                ║
║     3. Fourchette étroite : σ fraude = 21k vs 110k légit    ║
║     4. Soldes destination 2.6× plus faibles dans les fraudes║
║     5. Taux fraude varie 4% → 18% selon la période          ║
║     6. Split temporel OBLIGATOIRE                           ║
║                                                              ║
║  ⚙️  FEATURE ENGINEERING                                     ║
║     29 nouvelles features créées                             ║
║     35 features finales pour la modélisation                 ║
║                                                              ║
║  📁 DONNÉES PRÊTES                                           ║
║     X_train : 1 035 820 × 35  (taux fraude 9.10%)          ║
║     X_val   :   254 261 × 35  (taux fraude 13.86%)         ║
║     X_test  :   430 100 × 35  (à prédire)                  ║
║                                                              ║
║  ✅ MODÈLES RECOMMANDÉS                                      ║
║     Principal  : LightGBM (rapide, précis)                  ║
║     Secondaire : XGBoost (ensemble)                         ║
║     Baseline   : Régression Logistique                      ║
╚══════════════════════════════════════════════════════════════╝
''')

---
## 🚀 Prochaines Étapes (hors scope de ce notebook)

Ce notebook couvre l'**EDA et la préparation des données**. La modélisation sera développée séparément.

### Plan de modélisation recommandé :

```python
# 1. Baseline
from sklearn.linear_model import LogisticRegression
baseline = LogisticRegression(class_weight='balanced')
# → Attendu : ROC-AUC ≈ 0.75-0.82

# 2. Modèle principal
import lightgbm as lgb
lgbm = lgb.LGBMClassifier(is_unbalance=True, learning_rate=0.05)
# → Attendu : ROC-AUC ≈ 0.90-0.97

# 3. Ensemble
y_pred = 0.5 * lgbm.predict_proba(X_test)[:,1] + \
         0.5 * xgb.predict_proba(X_test)[:,1]
# → Toujours meilleur qu'un seul modèle
```

---

*📁 Fichiers produits : `eda_local.py` · `EDA_DOCUMENTATION.md` · `choix_modeles.md` · `notebook_colab.ipynb`*  
*Auteur : Équipe Octave | DataTour 2026 | Juin 2026*